In [18]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

## 1.初始化并调用模型

langchain提供了两种常见方法用来初始化模型：
- 使用init_chat_model方法，由langchain自动创建模型对象
- 使用不同模型对应的Model类，手动创建模型对象


### 1.1.init_chat_model
使用init_chat_model方法，langchain根据模型名称自动初始化与模型的连接，非常方便。

但需要注意的是：**如果使用我们必须在.env中配置好模型提供者的api_key**


In [19]:
# 导入Langchain的初始化模型的函数
import os
from langchain.chat_models import init_chat_model

# SiliconFlow 兼容 OpenAI 接口，所以需要显式指定 model_provider、base_url 和 api_key
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider="openai",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY"),
)

In [20]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.2.自定义模型

init_chat_model默认会根据模型名称自动确定模型的提供者、其base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问 .env 中配置的 OpenAI 兼容模型时，可以自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [21]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("BASE_URL")
api_key = os.getenv("SILICONFLOW_API_KEY")

model = init_chat_model(
    model=os.getenv("MODEL_NAME"),  # 模型名称，从 .env 中读取
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [22]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.3.调整模型参数
除了修改模型提供者以外，init_chat_model方法允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [23]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),  # 模型名称，从 .env 中读取
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5,
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.4.使用model类
其实init_chat_model方法底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [24]:
from langchain_openai import ChatOpenAI

# 使用Model类初始化模型
model = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY"),
    # 其它模型参数...
)

In [25]:
# 打印结果
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


# 2.访问模型

LangChain提供了两个不同的方法来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 2.1.invoke
invoke方法是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [26]:
# 通过invoke方法访问模型，需要阻塞等待模型生成结果
response = model.invoke("月亮的首都是哪里？")

In [27]:
# 查看响应内容
print(response)

content='\n\n月亮作为地球的天然卫星，本身没有国家或行政区域，因此不存在“首都”这一概念。不过，人类对月球的探索和想象中，有一些有趣的点可以提及：\n\n1. **科学探索**：目前，月球上没有常住人口或城市，但多个国家和组织（如美国、中国、俄罗斯等）已经开展了月球探测任务，未来可能建立月球基地，但这些仍属于科研或实验性质，不是“首都”。\n\n2. **虚构设定**：在一些科幻作品中，月球可能被设定为某个虚构国家或殖民地的中心，例如：\n   - 《月球》（电影）中的太空站“Lunar Colony”。\n   - 《星球大战》中的“月球城市”（如Corulian或某些设定中的月球殖民地）。\n   - 一些小说或游戏中可能会将月球作为“月球国”的首都。\n\n3. **文化象征**：在中国古代传说中，月亮常与“月宫”“广寒宫”等神话关联，称为嫦娥居住的地方，但这并非真实存在的地点。\n\n4. **可能的误解**：若问题有误（如混淆“月球”与“地球上的某个地点”），可以进一步澄清。例如，中国曾将“月球”作为某些项目的象征性名称，但并无实际意义。\n\n若您是指月球上的某个特定区域或虚构设定，可能需要更具体的背景信息哦！ 😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 811, 'prompt_tokens': 14, 'total_tokens': 825, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 501, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'openai', 'model_name': 'Qwen/Qw

## 2.2.流式访问

阻塞式调用需要等待较长时间才能看到AI返回的结果，而流式调用则可以实时看到AI返回的一个个词。

In [28]:
# 通过.stream方法实现流式访问
stream = model.stream("月亮的首都是哪里？")

# 流式访问会返回一个生成器对象，可以遍历生成器对象来实时获取AI的回复
print(type(stream))

<class 'generator'>


In [29]:
# 遍历stream结果，实时打印AI的回复
for chunk in stream:
    print(chunk.content, end="", flush=True)



“月亮的首都是哪里？”这个问题本身存在一些概念上的混淆，因为月亮（Moon）是地球的天然卫星，是一个天体，不具备国家、城市或行政机构的属性，因此没有“首都”这一说法。

不过，可能有以下几种不同的理解方向，可以逐一澄清：

---

### 1. **科学角度：月球的“基地”**
   - 目前人类尚未在月球上建立永久的、具有行政职能的“首都”或城市。
   - 唯一的人类月球基地是**阿波罗计划**期间（1960-1972年）美国宇航局（NASA）在月球表面设立的临时科研站，例如著名的**阿波罗11号着陆点**（静海基地）。但这些是短暂的科学实验点，而非国家或地区的首都。
   - 一些国家（如美国、中国、俄罗斯、日本等）计划未来在月球建立永久基地，例如中国“嫦娥计划”中的月球科研站，但这些仍属于科研设施，目前没有“首都”的定义。

---

### 2. **神话或文化角度**
   - 在中国神话中，月亮上居住着**嫦娥**和**玉兔**，但这些是虚构的传说，没有实际的“首都”或行政中心。
   - 在希腊神话中，月亮神是**阿尔忒弥斯**（Artemis），她的神殿位于**德尔斐**（希腊）或**奥林匹斯山**（神话中的神之居所），但这也与现实中的“首都”无关。
   - 若从文学或艺术作品延伸，例如某些小说或游戏可能虚构“月亮之城”或“月球帝国”，但这些属于虚构设定，需根据具体作品判断。

---

### 3. **地理名称的误用**
   - 地球上有一些地名可能与“月亮”相关，例如：
     - **月牙泉**（中国甘肃敦煌）：一个因形状如弯月而得名的湖泊。
     - **月亮湾**（中国多地有同名地点）：通常指河湾或海岸的弯曲地貌。
     - **月球村**（Lunarpedia）：指人类未来可能建立的月球定居点，但这是概念性称呼，非正式地名。
   - 此外，某些国家或地区的城市可能因与月亮的关联而被特别称为“月亮之城”，但这类名称大多属于地方特色，并非“首都”。

---

### 4. **可能的误解或玩笑**
   - 如果这是个玩笑或比喻，可能指代某个以“月亮”命名的地点，但需结合具体语境。例如：
     - **月球上的某个地名**：尽管月球表面有以科学家或历史人物命名的陨石坑（如“哥白尼环形山”），但这些并非“首都”。

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent方法用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [30]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [31]:
# 2.使用 SiliconFlow 模型对象创建 Agent
agent = create_agent(model=model)

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [32]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "月亮的首都是哪里？"}]
})

print(response)

{'messages': [HumanMessage(content='月亮的首都是哪里？', additional_kwargs={}, response_metadata={}, id='52555f9c-f850-48c7-8f6f-0d833bc66b53'), AIMessage(content='\n\n月亮（月球）是一个自然天体，没有政治实体或人类居住的首都。你提到的“月亮的首都”可能是一个虚构的概念、比喻，或者对“月球基地”的误解。\n\n不过，如果从其他角度理解，可以提到以下几点：\n\n1. **广寒宫**：在中国古代神话中，月球上有一座“广寒宫”，是嫦娥仙子居住的地方，但这只是神话传说，并非现实中的地点。\n\n2. **月球上的地标**：  \n   - **静海基地（Tranquility Base）**：这是阿波罗11号登月任务的着陆点，位于月球正面的静海（Sea of Tranquility），是人类首次登月的地点，但并非“首都”。  \n   - **其他人类探测器命名**：例如“月球车”、“月球轨道器”等，但这些也属于科研设施，而非政治中心。\n\n3. **科幻设定**：在某些科幻作品中（如《三体》《星际穿越》等），月亮可能被赋予特殊意义或虚构的“首都”概念，但这属于虚构创作。\n\n若你指的是地球上的某个地方被称为“月亮的首都”，可能需要更多背景信息以便更准确地解答。例如，某些文化中可能有与月亮相关的象征性地点，但通常不称为“首都”。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 711, 'prompt_tokens': 14, 'total_tokens': 725, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 422, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_token

### 流式访问


In [33]:
# 通过stream方法实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "月亮的首都是哪里？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [34]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token



“月亮的首都是哪里？”这个问题本身存在一些概念上的混淆，因为**月球并非一个国家或独立的政治实体**，因此没有“首都”这一说法。以下是几种可能的解释和回答方向：

---

### 1. **如果指月球本身：**
   - 月球是地球的天然卫星，目前没有被任何国家正式管辖（根据《外层空间条约》，月球属于全人类）。因此它不构成一个国家，也就没有首都。
   - **人类对月球的探索**：目前NASA、中国国家航天局（CNSA）等机构在月球上建立了实验性基地或研究站（如美国的“阿波罗计划”遗址、中国的“嫦娥三号”着陆点），但这些只是科研设施，并非“首都”。

---

### 2. **如果指某个以“月亮”为别称的国家：**
   - 有些国家或地区因文化、地理等原因被称为“月亮之国”：
     - **不丹**：传统上被称作“月亮之国”（Lingkhor），但这是文化象征，并非正式名称。
     - **其他可能性**：历史上某些地区（如古巴、缅甸）曾因文化或宗教被赋予类似称呼，但需具体背景确认。

   **建议**：如果这是指某个特定国家的别称，可以补充说明其背景，但需用户提供更多信息。

---

### 3. **如果指文学、影视或游戏中的虚构设定：**
   - 在科幻作品中，月球可能被设定为某个国家或文明的所在地。例如：
     - 美国作家**弗兰克·赫伯特**的《沙丘》系列中，有虚构的“月亮”（但实际是另一颗星球）。
     - 游戏或小说中的设定可能需要具体作品名称。

   **建议**：如果涉及虚构内容，请提供更多上下文以便准确回答。

---

### 4. **如果存在语言或文化误解：**
   - 可能是中文谐音或误译。例如，“月亮”在某些方言或语境中可能被用作其他含义，但通常需要更多线索才能确认。

---

### 总结回答：
**“月球没有首都。”** 它是地球的卫星，目前未被任何国家宣称主权。若指其他文化或虚构中的“月亮之国”，请提供更多背景信息。如果是比喻或调侃，也可以解释为“月球的‘首都’可能是人类未来建立的首个月球基地，比如月球南极的科研站”。